# Scraper for Regulations and Guidance

Link: https://www.mas.gov.sg/regulation/regulations-and-guidance?topics=Anti-Money%20Laundering&page=1&rows=All

## Obtain All the Links

In [1]:
!pip install pdf-diff

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 29.4 MB/s eta 0:00:00a 0:00:01
  DEPRECATION: Building 'diff-match-patch-python' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'diff-match-patch-python'. Discussion can be found at https://github.com/pypa/pip/issues/6334
  Created wheel for diff-match-patch-python: filename=diff_match_patch_python-1.0.3-cp310-cp310-macosx_11_0_arm64.whl size=59414 sha256=8b4abdd551f5b4e934b9f4aebc2263808cdbc0ae3c2e0b68efebfcdb37a93251
  Stored in directory: /Users/javianng/Library/Caches/pip/wheels/a9/1a/85/c1e06ad5de58d871265b0f35a80f32115e09a0b9b7dbadb16b
Successfully built diff-match-patch-python
   ━━━━━━━━━━━

In [ ]:
"""
MAS Regulatory Pages Scraper Utilities
"""
import re
from typing import List, Dict
from bs4 import BeautifulSoup
from selenium import webdriver
from urllib.parse import urljoin
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager


def mas_regulations_scraper(url: str) -> List[Dict]:
    """
    Scrapes regulations and guidance from MAS search results page.

    Args:
        url: The URL of the MAS regulations and guidance search page

    Returns:
        A list of regulation items, each containing:
        - title: Title of the regulation
        - url: Link to the regulation page
        - category: Category/tag of the regulation
        - date: Publication/update date
        - summary: Brief summary
        - topics: List of related topics
        - consultation_fields: Optional consultation information

    Example:
        >>> items = mas_regulations_scraper("https://www.mas.gov.sg/regulation/regulations-and-guidance?topics=Anti-Money%20Laundering&page=1&rows=All")
        >>> print(len(items))
        136
    """
    # Setup Chrome driver
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()))
    driver.get(url)
    html_content = driver.page_source
    driver.quit()

    # Prettify the HTML content
    soup = BeautifulSoup(html_content, 'html.parser')

    base_url = "https://www.mas.gov.sg"

    items = []
    for li in soup.find_all('li', class_='mas-search-page__result'):
        # title + link
        a = li.select_one('.ola-field-title a.mas-link') or li.find('a', class_='mas-link')
        title = a.get_text(" ", strip=True) if a else None
        href = urljoin(base_url, a['href']) if a and a.has_attr('href') else None

        # category / tag
        tag_el = li.select_one('.mas-tag__text')
        category = tag_el.get_text(strip=True) if tag_el else None

        # date (try to find a DD Month YYYY pattern inside ancillaries)
        date = None
        anc = li.select_one('.mas-ancillaries')
        if anc:
            text = anc.get_text(" ", strip=True)
            m = re.search(r'\d{1,2}\s+[A-Za-z]+\s+\d{4}', text)
            date = m.group(0) if m else text.strip()

        # summary / body
        body_p = li.select_one('.mas-search-card__body p')
        summary = body_p.get_text(" ", strip=True) if body_p else None

        # topics / footer tags (may be multiple)
        topics = []
        for foot_a in li.select('footer a.mas-link .mas-link__text'):
            t = foot_a.get_text(" ", strip=True)
            if t and t not in topics:
                topics.append(t)

        # consultation fields (optional)
        consultation = {}
        for cf in li.select('.consultation-field'):
            label = cf.contents[0].strip() if cf.contents else ''
            span = cf.select_one('span')
            if span:
                consultation[label.rstrip(':')] = span.get_text(" ", strip=True)

        items.append({
            "title": title,
            "url": href,
            "category": category,
            "date": date,
            "summary": summary,
            "topics": topics,
            "consultation_fields": consultation or None
        })

    return items

## Extract History

In [ ]:
"""
Notice History Scraper for MAS Regulatory Pages
"""

from typing import List, Dict
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium. webdriver. chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

def notice_history_scraper(url: str) -> List[Dict]:
    """
    Scrapes the amendment history from a MAS notice page.

    Args:
        url: The URL of the MAS notice page to scrape

    Returns:
        A list of amendment entries, each containing:
        - date: The date of the amendment
        - documents: List of documents with title and url

    Example:
        >>> entries = notice_history_scraper("https://www.mas.gov.sg/regulation/notices/notice-314")
        >>> print(entries)
        [
            {
                "date": "01 Jan 2024",
                "documents": [
                    {
                        "title": "Amendment Notice",
                        "url": "/path/to/document.pdf"
                    }
                ]
            }
        ]
    """
    # Setup Chrome driver
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()))
    driver.get(url)
    html_content = driver.page_source
    driver.quit()

    soup = BeautifulSoup(html_content, 'html.parser')

    # Find the description list containing amendment notes
    dl = soup.find('dl', class_='mas-description-list')

    if not dl:
        return []

    # Find all div elements that contain dt/dd pairs
    amendment_entries = []

    for div in dl.find_all('div', recursive=False):
        dt = div.find('dt')
        dd = div.find('dd')

        if dt and dd:
            # Extract date
            date = dt.get_text(strip=True)

            # Extract all PDF links from dd
            documents = []
            for link in dd.find_all('a', class_='mas-link'):
                # Extract title
                title_span = link.find('span', class_='mas-link__text')
                title = title_span.get_text(strip=True) if title_span else ""

                # Extract URL
                doc_url = link.get('href', '')

                document = {
                    "title": title,
                    "url": doc_url
                }

                documents.append(document)

            # Only add entry if it has documents
            if documents:
                amendment_entries.append({
                    "date": date,
                    "documents": documents
                })

    return amendment_entries

In [60]:
mas_regulations_scraper("https://www.mas.gov.sg/regulation/regulations-and-guidance?rows=10&sort=mas_date_tdt%20desc&page=1&topics=Anti-Money%20Laundering")

[{'title': 'Notice 314 Prevention of Money Laundering and Countering the Financing of Terrorism – Life Insurers',
  'url': 'https://www.mas.gov.sg/regulation/notices/notice-314',
  'category': 'Notices',
  'date': '30 June 2025',
  'summary': 'Requirements for direct life insurers to exercise due diligence and conduct their business with high ethical standards, to guard against money laundering and terrorism financing.',
  'topics': ['AML/CFT'],
  'consultation_fields': None},
 {'title': 'Notice TCA-N03 Prevention of Money Laundering and Countering the Financing of Terrorism - Trust Companies',
  'url': 'https://www.mas.gov.sg/regulation/notices/notice-tca-n03',
  'category': 'Notices',
  'date': '30 June 2025',
  'summary': 'Requirements for trust companies on anti-money laundering (AML) and countering the financing of terrorism (CFT).',
  'topics': ['AML/CFT'],
  'consultation_fields': None},
 {'title': 'Notice SFA 04-N20 to Specified Licence Holders and Specified Exempt Persons in r

In [59]:
notice_history_scraper("https://www.mas.gov.sg/regulation/notices/notice-314")

[{'date': '30 Jun 2025',
  'documents': [{'title': 'MAS Notice 314 (Cancellation) Notice 2025',
    'url': '/-/media/mas-media-library/regulation/notices/id/notice-314/mas-notice-314-cancellation-notice-2025.pdf'},
   {'title': 'MAS Notice 314 dated 24 Apr 2015 (last revised 1 March 2022)',
    'url': '/-/media/mas-media-library/regulation/notices/id/notice-314/notice-314-last-revised-on-1-march-2022.pdf'}]},
 {'date': '01 Mar 2022',
  'documents': [{'title': 'Notice 314 (Amendment) 2022',
    'url': '/-/media/mas-media-library/regulation/notices/id/notice-314/notice-314-amendment-2022.pdf'}]},
 {'date': '28 Jun 2021',
  'documents': [{'title': 'Notice 314 (Amendment) 2021',
    'url': '/-/media/mas-media-library/regulation/notices/id/notice-314/mas-314_tracked_28-jun-2021.pdf'}]},
 {'date': '30 Nov 2015',
  'documents': [{'title': 'Notice 314 (Amendment) 2015',
    'url': '/-/media/mas/regulations-and-financial-stability/regulatory-and-supervisory-framework/anti_money-laundering_count

# Full Running

In [ ]:
for regulation in mas_regulations_scraper("https://www.mas.gov.sg/regulation/regulations-and-guidance?rows=10&sort=mas_date_tdt%20desc&page=1&topics=Anti-Money%20Laundering"): # this is in json format
    if regulation['category'] == 'Notices':
        notice_history_scraper(regulation['url'])
    elif regulation['category'] == 'Guidelines':
        notice_history_scraper(regulation['url'])
    else: 
        pass